# BraTS2020 sequence classification with frozen BrainIAC

This notebook runs the `crop_pad_zscore` pipeline on a Colab GPU while keeping the BrainIAC backbone frozen. It caches features, trains only the classifier head, then generates the visual debugging outputs.

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Upload these two files to /content before starting the run:
# 1) brainiac_colab_runtime_bundle.zip
# 2) archive.zip
bundle_path = Path('/content/brainiac_colab_runtime_bundle.zip')
uploaded_archive_path = Path('/content/archive.zip')
repo_root = Path('/content/BrainIAC')
repo_archive_path = repo_root / 'archive.zip'

if bundle_path.exists() and not repo_root.exists():
    with zipfile.ZipFile(bundle_path) as bundle:
        bundle.extractall('/content')
    print('extracted bundle:', bundle_path)

if uploaded_archive_path.exists() and not repo_archive_path.exists():
    repo_root.mkdir(parents=True, exist_ok=True)
    shutil.move(str(uploaded_archive_path), str(repo_archive_path))
    print('moved archive:', uploaded_archive_path, '->', repo_archive_path)

print(repo_root, 'ok' if repo_root.exists() else 'missing')
print(repo_root / 'requirements.txt', 'ok' if (repo_root / 'requirements.txt').exists() else 'missing')
print(repo_archive_path, 'ok' if repo_archive_path.exists() else 'missing')


In [ ]:
from pathlib import Path
import subprocess
import sys

# Edit these only if your Colab paths differ.
REPO_ROOT = Path('/content/BrainIAC')
PROJECT_ROOT = REPO_ROOT / 'brats_sequence_project'
BRAINIAC_SRC = REPO_ROOT / 'src'
CHECKPOINT_CANDIDATES = [
    Path('/content/checkpoints/BrainIAC.ckpt'),
    REPO_ROOT / 'src/checkpoint/BrainIAC.ckpt',
    REPO_ROOT / 'src/checkpoints/BrainIAC.ckpt',
]
CHECKPOINT_PATH = next((path for path in CHECKPOINT_CANDIDATES if path.exists()), CHECKPOINT_CANDIDATES[0])
DATASET_ROOT = Path('/content/Dataset')
ARCHIVE_ZIP = REPO_ROOT / 'archive.zip'

VARIANT = 'crop_pad_zscore'
FEATURE_BATCH_SIZE = 4
FEATURE_WORKERS = 2
CLASSIFIER_EPOCHS = 30
CLASSIFIER_BATCH_SIZE = 32

TRAIN_CSV = PROJECT_ROOT / 'outputs/train.csv'
VAL_CSV = PROJECT_ROOT / 'outputs/val.csv'
TEST_CSV = PROJECT_ROOT / 'outputs/test.csv'
ABLATION_ROOT = PROJECT_ROOT / 'outputs/preprocessing_ablation' / VARIANT
FEATURES_DIR = ABLATION_ROOT / 'features'
CLASSIFIER_DIR = ABLATION_ROOT / 'classifier'
VISUAL_DEBUG_DIR = PROJECT_ROOT / 'outputs/visual_debug'

for path in [PROJECT_ROOT, BRAINIAC_SRC, CHECKPOINT_PATH, TRAIN_CSV, VAL_CSV, TEST_CSV, DATASET_ROOT, ARCHIVE_ZIP]:
    print(f'{path}:', 'ok' if path.exists() else 'missing')


In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
print('gpu_name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Install the repo requirements inside the Colab runtime if needed.
requirements_path = REPO_ROOT / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'requirements.txt not found: {requirements_path}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)


In [ ]:
# The CSVs were generated locally and still contain /Users/kp/... image paths.
# If the extracted Dataset tree exists, create a Colab-only compatibility symlink.
# If only archive.zip exists, BraTSSequenceDataset will use its built-in archive fallback.
LEGACY_DATASET_ROOT = Path('/Users/kp/Documents/BRAINIAC/BrainIAC/Dataset')
if DATASET_ROOT.exists():
    LEGACY_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if LEGACY_DATASET_ROOT.exists() or LEGACY_DATASET_ROOT.is_symlink():
        print('legacy dataset path already exists:', LEGACY_DATASET_ROOT)
    else:
        LEGACY_DATASET_ROOT.symlink_to(DATASET_ROOT)
        print('created symlink:', LEGACY_DATASET_ROOT, '->', DATASET_ROOT)
elif ARCHIVE_ZIP.exists():
    print('using archive fallback:', ARCHIVE_ZIP)
else:
    raise FileNotFoundError(f'Neither extracted dataset nor archive fallback exists: {DATASET_ROOT}, {ARCHIVE_ZIP}')

missing = [path for path in [PROJECT_ROOT, BRAINIAC_SRC, CHECKPOINT_PATH, TRAIN_CSV, VAL_CSV, TEST_CSV] if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths: ' + ', '.join(map(str, missing)))


In [ ]:
def run(cmd):
    printable = ' '.join(map(str, cmd))
    print('\n$', printable)
    subprocess.run([str(part) for part in cmd], cwd=PROJECT_ROOT, check=True)

python = sys.executable


## 1. Pre-run visual checks

In [ ]:
run([
    python, 'scripts/visualize_preprocessed_batch.py',
    '--csv_path', TRAIN_CSV,
    '--variant', VARIANT,
    '--batch_size', '20',
    '--split_name', 'train',
    '--output_dir', VISUAL_DEBUG_DIR / 'train_batch_crop',
])
run([
    python, 'scripts/compare_preprocessing_visuals.py',
    '--csv_path', TRAIN_CSV,
    '--variants', 'resize_zscore', VARIANT,
    '--output_dir', VISUAL_DEBUG_DIR / 'compare_resize_vs_crop',
])
run([python, 'scripts/audit_brainiac_paper_repo_alignment.py'])


## 2. Cache frozen BrainIAC features

In [ ]:
for split_name, csv_path in [('train', TRAIN_CSV), ('val', VAL_CSV), ('test', TEST_CSV)]:
    run([
        python, 'scripts/cache_brainiac_features.py',
        '--brainiac_src', BRAINIAC_SRC,
        '--checkpoint_path', CHECKPOINT_PATH,
        '--csv_path', csv_path,
        '--split_name', split_name,
        '--variant', VARIANT,
        '--output_dir', FEATURES_DIR,
        '--batch_size', str(FEATURE_BATCH_SIZE),
        '--num_workers', str(FEATURE_WORKERS),
        '--use_amp',
    ])


## 3. Train only the classifier head

In [ ]:
run([
    python, 'scripts/train_cached_feature_classifier.py',
    '--train_features', FEATURES_DIR / f'features_train_{VARIANT}.pt',
    '--val_features', FEATURES_DIR / f'features_val_{VARIANT}.pt',
    '--test_features', FEATURES_DIR / f'features_test_{VARIANT}.pt',
    '--output_dir', CLASSIFIER_DIR,
    '--epochs', str(CLASSIFIER_EPOCHS),
    '--lr', '1e-3',
    '--weight_decay', '1e-4',
    '--batch_size', str(CLASSIFIER_BATCH_SIZE),
    '--seed', '42',
    '--patience', '5',
])


## 4. Generate prediction visualizations

In [ ]:
VAL_PREDICTIONS = CLASSIFIER_DIR / 'val_predictions.csv'
TEST_PREDICTIONS = CLASSIFIER_DIR / 'predictions.csv'

run([
    python, 'scripts/visualize_batch_with_predictions.py',
    '--csv_path', VAL_CSV,
    '--predictions_csv', VAL_PREDICTIONS,
    '--variant', VARIANT,
    '--batch_size', '20',
    '--output_dir', VISUAL_DEBUG_DIR / 'val_batch_predictions',
])
run([
    python, 'scripts/inspect_validation_outputs.py',
    '--predictions_csv', VAL_PREDICTIONS,
    '--output_dir', VISUAL_DEBUG_DIR / 'validation_output_inspection',
])
run([
    python, 'scripts/visualize_prediction_errors.py',
    '--csv_path', TEST_CSV,
    '--predictions_csv', TEST_PREDICTIONS,
    '--variant', VARIANT,
    '--output_dir', VISUAL_DEBUG_DIR / 'error_analysis',
])


## 5. Inspect key outputs

In [ ]:
important_outputs = [
    CLASSIFIER_DIR / 'best_metrics.json',
    CLASSIFIER_DIR / 'test_metrics.json',
    CLASSIFIER_DIR / 'val_predictions.csv',
    CLASSIFIER_DIR / 'predictions.csv',
    VISUAL_DEBUG_DIR / 'train_batch_crop' / f'batch_train_{VARIANT}_20cases.png',
    VISUAL_DEBUG_DIR / 'val_batch_predictions' / f'validation_batch_predictions_{VARIANT}.png',
    VISUAL_DEBUG_DIR / 'validation_output_inspection' / 'confidence_correct_vs_incorrect.png',
    VISUAL_DEBUG_DIR / 'error_analysis' / 'wrong_predictions_grid.png',
]
for path in important_outputs:
    print(path, 'ok' if path.exists() else 'missing')
